In [1]:
"""
This train/eval notebook contains the new iteration of the
CORAL experiment in which source data is adapted to target
characteristics (source to target) and classifiers fitted 
on both the original and adapted source data are evaluated
on the target.
"""
import pandas as pd
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, recall_score
import joblib
import os

In [2]:
# Load datasets and pretrained artifacts
source_train_data_path = os.path.join('data', 'processed', 'source', 'train.csv')
source_test_data_path = os.path.join('data', 'processed', 'source', 'test.csv')
target_train_data_path = os.path.join('data', 'processed', 'target', 'train.csv')
target_test_data_path = os.path.join('data', 'processed', 'target', 'test.csv')

label_encoder_path = os.path.join('models', 'label_encoder.joblib')
coral_source_stats_path = os.path.join('models', 'coral_source_stats.joblib')
coral_target_stats_path = os.path.join('models', 'coral_target_stats.joblib')

source_train_df = pd.read_csv(source_train_data_path)
source_test_df = pd.read_csv(source_test_data_path)
target_train_df = pd.read_csv(target_train_data_path)
target_test_df = pd.read_csv(target_test_data_path)

label_encoder = joblib.load(label_encoder_path)
coral_source_stats = joblib.load(coral_source_stats_path)
coral_target_stats = joblib.load(coral_target_stats_path)

print(f"Source train shape: {source_train_df.shape}")
print(f"Source test shape: {source_test_df.shape}")
print(f"Target train shape: {target_train_df.shape}")
print(f"Target test shape: {target_test_df.shape}")
print("Loaded artifacts: label_encoder, coral_source_stats, coral_target_stats")

Source train shape: (1072115, 78)
Source test shape: (268029, 78)
Target train shape: (1856679, 78)
Target test shape: (464170, 78)
Loaded artifacts: label_encoder, coral_source_stats, coral_target_stats


In [3]:
# Sanity checks
# Verify source and target datasets have equivalent feature and label space.

shared_feature_path = os.path.join('data', 'processed', 'shared_feature_space.json')
shared_label_path = os.path.join('data', 'processed', 'shared_label_space.json')

import json
with open(shared_feature_path, 'r') as f:
    shared_feature_payload = json.load(f)
with open(shared_label_path, 'r') as f:
    shared_label_payload = json.load(f)

# Accept either list format or object payload for schema compatibility with preprocessing notebooks.
if isinstance(shared_feature_payload, dict):
    if 'features' not in shared_feature_payload:
        raise ValueError("Expected key 'features' when shared feature payload is a JSON object.")
    shared_features = list(shared_feature_payload['features'])
elif isinstance(shared_feature_payload, list):
    shared_features = list(shared_feature_payload)
else:
    raise ValueError(f"Unsupported shared feature payload type: {type(shared_feature_payload).__name__}")

if isinstance(shared_label_payload, dict):
    if 'labels' not in shared_label_payload:
        raise ValueError("Expected key 'labels' when shared label payload is a JSON object.")
    shared_labels = list(shared_label_payload['labels'])
elif isinstance(shared_label_payload, list):
    shared_labels = list(shared_label_payload)
else:
    raise ValueError(f"Unsupported shared label payload type: {type(shared_label_payload).__name__}")

assert set(shared_features) <= set(source_train_df.columns), "Source train missing shared features!"
assert set(shared_features) <= set(source_test_df.columns), "Source test missing shared features!"
assert set(shared_features) <= set(target_train_df.columns), "Target train missing shared features!"
assert set(shared_features) <= set(target_test_df.columns), "Target test missing shared features!"

# Normalize each split's labels into class-name space before set comparison.
def to_label_name_set(label_series, fitted_label_encoder):
    labels = label_series.dropna()
    if pd.api.types.is_numeric_dtype(labels):
        return set(fitted_label_encoder.inverse_transform(labels.astype(int).to_numpy()))
    return set(labels.astype(str).to_numpy())

expected_label_set = set(map(str, shared_labels))
source_train_label_set = to_label_name_set(source_train_df['Label'], label_encoder)
source_test_label_set = to_label_name_set(source_test_df['Label'], label_encoder)
target_train_label_set = to_label_name_set(target_train_df['Label'], label_encoder)
target_test_label_set = to_label_name_set(target_test_df['Label'], label_encoder)

source_combined_label_set = source_train_label_set | source_test_label_set
target_combined_label_set = target_train_label_set | target_test_label_set

assert source_combined_label_set == expected_label_set, (
    "Combined source label space mismatch vs shared labels! "
    f"Missing={sorted(expected_label_set - source_combined_label_set)}, "
    f"Extra={sorted(source_combined_label_set - expected_label_set)}"
 )
assert target_combined_label_set == expected_label_set, (
    "Combined target label space mismatch vs shared labels! "
    f"Missing={sorted(expected_label_set - target_combined_label_set)}, "
    f"Extra={sorted(target_combined_label_set - expected_label_set)}"
 )
assert source_combined_label_set == target_combined_label_set, (
    "Combined source/target label spaces do not match!"
 )

print("Sanity checks passed: Shared feature space verified for source train/test and target train/test.")
print("Sanity checks passed: Combined source and target label spaces match shared labels.")

Sanity checks passed: Shared feature space verified for source train/test and target train/test.
Sanity checks passed: Combined source and target label spaces match shared labels.


In [4]:
### Calculate CORAL transform and apply to both source splits (train and test) ###


def stable_symmetric_matrix_power(matrix, power, eps=1e-6):
    """Return a numerically stabilized symmetric matrix power."""
    matrix = np.asarray(matrix, dtype=np.float64)
    matrix = (matrix + matrix.T) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(matrix)
    ridge = max(eps, float(-eigenvalues.min() + eps)) if eigenvalues.min() <= 0 else eps
    clipped_eigenvalues = np.clip(eigenvalues + ridge, eps, None)

    powered = eigenvectors @ np.diag(clipped_eigenvalues ** power) @ eigenvectors.T
    return (powered + powered.T) / 2.0, float(eigenvalues.min()), ridge


# Validate stats and feature ordering before applying CORAL.
source_feature_order = coral_source_stats.get("feature_order")
target_feature_order = coral_target_stats.get("feature_order")

if source_feature_order is None or target_feature_order is None:
    raise KeyError("CORAL stats must include 'feature_order' metadata.")
if list(source_feature_order) != list(target_feature_order):
    raise ValueError("Source and target CORAL stats feature_order do not match.")
if list(shared_features) != list(source_feature_order):
    raise ValueError(
        "Runtime shared feature order does not match CORAL stats feature_order. "
        "Regenerate stats or align feature ordering before evaluation."
    )

# Extract source/target statistics used by the source-to-target CORAL transform.
source_mean = np.asarray(coral_source_stats["mean"], dtype=np.float64)
source_cov = np.asarray(coral_source_stats["covariance"], dtype=np.float64)

target_mean = np.asarray(coral_target_stats["mean"], dtype=np.float64)
target_cov = np.asarray(coral_target_stats["covariance"], dtype=np.float64)

n_features = len(source_feature_order)
if source_mean.shape[0] != n_features or target_mean.shape[0] != n_features:
    raise ValueError("CORAL mean vector length does not match feature_order length.")
if source_cov.shape != (n_features, n_features) or target_cov.shape != (n_features, n_features):
    raise ValueError("CORAL covariance shape does not match feature_order length.")

# Convert source features to numpy using the validated canonical feature order.
X_source_train_np = source_train_df[source_feature_order].to_numpy(dtype=np.float64)
X_source_test_np = source_test_df[source_feature_order].to_numpy(dtype=np.float64)

# Center source data using source mean before CORAL transform.
X_source_train_centered = X_source_train_np - source_mean
X_source_test_centered = X_source_test_np - source_mean

# Build the source-to-target CORAL transform: A = Cs^(-1/2) * Ct^(1/2).
source_cov_inv_sqrt, source_min_eig, source_ridge = stable_symmetric_matrix_power(source_cov, -0.5)
target_cov_sqrt, target_min_eig, target_ridge = stable_symmetric_matrix_power(target_cov, 0.5)

A_coral = source_cov_inv_sqrt @ target_cov_sqrt

# Apply the same transform to both source train and source test splits.
X_source_train_coral_np = (X_source_train_centered @ A_coral) + target_mean
X_source_test_coral_np = (X_source_test_centered @ A_coral) + target_mean

if not np.isfinite(X_source_train_coral_np).all():
    raise ValueError("CORAL transform produced non-finite values for source train split.")
if not np.isfinite(X_source_test_coral_np).all():
    raise ValueError("CORAL transform produced non-finite values for source test split.")

X_source_train_coral_df = pd.DataFrame(
    X_source_train_coral_np,
    columns=source_feature_order,
    index=source_train_df.index,
)
X_source_test_coral_df = pd.DataFrame(
    X_source_test_coral_np,
    columns=source_feature_order,
    index=source_test_df.index,
)

print("CORAL source-to-target transform computed successfully.")
print(f"Feature count: {n_features}")
print(f"Source covariance min eig / ridge: {source_min_eig:.6e} / {source_ridge:.6e}")
print(f"Target covariance min eig / ridge: {target_min_eig:.6e} / {target_ridge:.6e}")
print(f"Adapted source train shape: {X_source_train_coral_df.shape}")
print(f"Adapted source test shape: {X_source_test_coral_df.shape}")

CORAL source-to-target transform computed successfully.
Feature count: 77
Source covariance min eig / ridge: 1.847600e-02 / 1.000000e-06
Target covariance min eig / ridge: 5.529764e+02 / 1.000000e-06
Adapted source train shape: (1072115, 77)
Adapted source test shape: (268029, 77)


In [ ]:
### Fit classifiers on original (no-CORAL) source train split ###


### Random Forest ###
# hyperparameters: n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=1
from sklearn.ensemble import RandomForestClassifier

X_source_train = source_train_df[shared_features]
y_source_train = source_train_df['Label'].astype(int)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
 )
rf.fit(X_source_train, y_source_train)

rf_model_path = os.path.join('models', 'rf_model.joblib')
joblib.dump(rf, rf_model_path)

print('Random Forest trained on source train split.')
print(f'Saved Random Forest model to: {rf_model_path}')

Random Forest trained on source train split.
Saved Random Forest model to: models/rf_no_coral_model.joblib


In [6]:
### Fit classifiers on adapted (with-CORAL) source train split ###

### Random Forest ###
# hyperparameters: n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=1
from sklearn.ensemble import RandomForestClassifier

X_source_train_coral = X_source_train_coral_df
y_source_train_coral = source_train_df['Label'].astype(int)

rf_coral = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
 )
rf_coral.fit(X_source_train_coral, y_source_train_coral)

rf_coral_model_path = os.path.join('models', 'rf_coral_model.joblib')
joblib.dump(rf_coral, rf_coral_model_path)

print('Random Forest trained on source train split.')
print(f'Saved Random Forest model to: {rf_coral_model_path}')

Random Forest trained on source train split.
Saved Random Forest model to: models/rf_coral_model.joblib


In [10]:
### Evaluate no-CORAL source-trained classifiers ###

X_source_test = source_test_df[shared_features]
y_source_test = source_test_df['Label'].astype(int)
X_target_test = target_test_df[shared_features]
y_target_test = target_test_df['Label'].astype(int)

print('==============================================')
print('RANDOM FOREST PERFORMANCE (NO CORAL TRAINING)')
print('==============================================')

# Evaluate the non-adapted-source-trained RF on the non-adapted source
# test split.

y_pred_source = rf_no_coral.predict(X_source_test)
source_accuracy = accuracy_score(y_source_test, y_pred_source)
source_macro_recall = recall_score(y_source_test, y_pred_source, average='macro', zero_division=0)

print('[S-trained RF on S_test]')
print(f'Accuracy: {source_accuracy:.6f}')
print(f'Macro Recall: {source_macro_recall:.6f}')
print('Classification Report:')
print(classification_report(y_source_test, y_pred_source, target_names=label_encoder.classes_, zero_division=0))

# Evaluate the non-adapted-source-trained RF on the target test split.

y_pred_target = rf_no_coral.predict(X_target_test)
target_accuracy = accuracy_score(y_target_test, y_pred_target)
target_macro_recall = recall_score(y_target_test, y_pred_target, average='macro', zero_division=0)

print('[S-trained RF on T_test]')
print(f'Accuracy: {target_accuracy:.6f}')
print(f'Macro Recall: {target_macro_recall:.6f}')
print('Classification Report:')
print(classification_report(y_target_test, y_pred_target, target_names=label_encoder.classes_, zero_division=0))

RANDOM FOREST PERFORMANCE (NO CORAL TRAINING)
[S-trained RF on S_test]
Accuracy: 0.998567
Macro Recall: 0.845329
Classification Report:
                            precision    recall  f1-score   support

                    Benign       1.00      1.00      1.00    201022
                       Bot       0.89      0.84      0.86       390
                      DDoS       1.00      1.00      1.00     25603
             DoS GoldenEye       1.00      1.00      1.00      2057
                  DoS Hulk       1.00      1.00      1.00     34569
          DoS Slowhttptest       1.00      1.00      1.00      1046
             DoS slowloris       1.00      0.99      0.99      1077
               FTP-Patator       1.00      1.00      1.00      1186
              Infiltration       1.00      0.86      0.92         7
               SSH-Patator       1.00      1.00      1.00       644
  Web Attack - Brute Force       0.71      0.93      0.81       294
Web Attack - Sql Injection       0.50      0.25

In [11]:
### Evaluate with-CORAL source-trained classifiers ###

# Evaluate the RF trained on the CORAL-adapted source train split.

X_source_test_coral = X_source_test_coral_df
y_source_test_coral = source_test_df['Label'].astype(int)
X_target_test = target_test_df[shared_features]
y_target_test = target_test_df['Label'].astype(int)

print('==============================================')
print('RANDOM FOREST PERFORMANCE (WITH CORAL TRAINING)')
print('==============================================')

# Evaluate the adapted-source-trained RF on the adapted source test split.

y_pred_source_coral = rf_coral.predict(X_source_test_coral)
source_accuracy_coral = accuracy_score(y_source_test_coral, y_pred_source_coral)
source_macro_recall_coral = recall_score(y_source_test_coral, y_pred_source_coral, average='macro', zero_division=0)

print('[S\'-trained RF on S\'_test]')
print(f'Accuracy: {source_accuracy_coral:.6f}')
print(f'Macro Recall: {source_macro_recall_coral:.6f}')
print('Classification Report:')
print(classification_report(y_source_test_coral, y_pred_source_coral, target_names=label_encoder.classes_, zero_division=0))

# Evaluate the adapted-source-trained RF on the target test split.

y_pred_target_coral = rf_coral.predict(X_target_test)
target_accuracy_coral = accuracy_score(y_target_test, y_pred_target_coral)
target_macro_recall_coral = recall_score(y_target_test, y_pred_target_coral, average='macro', zero_division=0)

print('[S\'-trained RF on T_test]')
print(f'Accuracy: {target_accuracy_coral:.6f}')
print(f'Macro Recall: {target_macro_recall_coral:.6f}')
print('Classification Report:')
print(classification_report(y_target_test, y_pred_target_coral, target_names=label_encoder.classes_, zero_division=0))

RANDOM FOREST PERFORMANCE (WITH CORAL TRAINING)
[S'-trained RF on S'_test]
Accuracy: 0.998336
Macro Recall: 0.793400
Classification Report:
                            precision    recall  f1-score   support

                    Benign       1.00      1.00      1.00    201022
                       Bot       0.85      0.81      0.83       390
                      DDoS       1.00      1.00      1.00     25603
             DoS GoldenEye       0.99      1.00      0.99      2057
                  DoS Hulk       1.00      1.00      1.00     34569
          DoS Slowhttptest       0.99      0.99      0.99      1046
             DoS slowloris       1.00      1.00      1.00      1077
               FTP-Patator       1.00      1.00      1.00      1186
              Infiltration       1.00      0.43      0.60         7
               SSH-Patator       0.99      0.99      0.99       644
  Web Attack - Brute Force       0.73      0.77      0.75       294
Web Attack - Sql Injection       0.00      